In [16]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_recall_fscore_support
from torch.utils.data import Dataset, DataLoader
from typing import Tuple, List, Dict

from utils import load_embeddings, tokenize_corpus, build_vocab_dict

torch.manual_seed(42)
np.random.seed(42)


In [17]:
class RNNClassifier(nn.Module):
    def __init__(self, embedding_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n.squeeze(0))


In [18]:
class SentenceDataset(Dataset):
    def __init__(self, X: torch.Tensor, y: torch.Tensor):
        self.X = X
        self.y = y
    
    def __len__(self) -> int:
        return len(self.X)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.X[idx], self.y[idx]

In [19]:
def sentence_to_embeddings(tokens: List[str], word_vecs: np.ndarray, vocab_dict: Dict[str, int], max_len: int) -> torch.Tensor:
    
    embeddings = []
    
    for word in tokens[:max_len]:
        try:
            word_idx = vocab_dict.index(word)
            embeddings.append(word_vecs[word_idx])
        except:
            pass
    
    if len(embeddings) == 0:
        return None
    
    embedding_dim = word_vecs.shape[1]
    while len(embeddings) < max_len:
        embeddings.append(np.zeros(embedding_dim))
    
    return torch.from_numpy(np.array(embeddings[:max_len])).float()


In [20]:
def prepare_dataset(data: pd.DataFrame, embeddings: Dict, regex_en: re.Pattern, regex_fr: re.Pattern, max_len: int) -> Tuple[torch.Tensor, torch.Tensor]:
    
    en_tokenized = tokenize_corpus(data['en'], regex_en)
    fr_tokenized = tokenize_corpus(data['fr'], regex_fr)
    
    sequences = []
    labels = []
    
    for en_tokens, fr_tokens in zip(en_tokenized, fr_tokenized):
        en_seq = sentence_to_embeddings(en_tokens, embeddings['en_vecs'], embeddings['en_words'], max_len)
        if en_seq is not None:
            sequences.append(en_seq)
            labels.append(0)
        
        fr_seq = sentence_to_embeddings(fr_tokens, embeddings['fr_vecs'], embeddings['fr_words'], max_len)
        if fr_seq is not None:
            sequences.append(fr_seq)
            labels.append(1)
    
    return torch.stack(sequences), torch.LongTensor(labels)


In [21]:
data = pd.read_csv("./data/french_english.tsv", sep='\t')
data.columns = ['id_en', 'en', 'id_fr', 'fr']
data = data.sample(n=5000, random_state=42).reset_index(drop=True)

regex_en = re.compile(r"[a-z]+")
regex_fr = re.compile(r"[a-zA-ZÀ-ÿ]+")

en_tokenized = tokenize_corpus(data['en'], regex_en)
fr_tokenized = tokenize_corpus(data['fr'], regex_fr)
max_len = max(max(len(s) for s in en_tokenized), max(len(s) for s in fr_tokenized))

print(f"Loaded {len(data)} sentence pairs, max length: {max_len}")

Loaded 5000 sentence pairs, max length: 82


In [22]:
emb_types = ['w2v', 'ft', 'glv']
datasets = {}

for emb_type in emb_types:
    print(f"Preparing {emb_type.upper()}...")
    embeddings = load_embeddings(emb_type)
    sequences, labels = prepare_dataset(data, embeddings, regex_en, regex_fr, max_len)
    
    # Split: 60% train, 20% val, 20% test
    seq_temp, seq_test, label_temp, label_test = train_test_split(sequences, labels, test_size=0.2, random_state=42, stratify=labels)
    seq_train, seq_val, label_train, label_val = train_test_split(seq_temp, label_temp, test_size=0.25, random_state=42, stratify=label_temp)
    
    datasets[emb_type] = {
        'train': (seq_train, label_train),
        'val': (seq_val, label_val),
        'test': (seq_test, label_test)
    }
    print(f"  Train: {len(seq_train)}, Val: {len(seq_val)}, Test: {len(seq_test)}")


Preparing W2V...
  Train: 5984, Val: 1995, Test: 1995
Preparing FT...
  Train: 5984, Val: 1995, Test: 1995
Preparing GLV...
  Train: 6000, Val: 2000, Test: 2000


In [23]:
def train(model, train_loader, val_loader, epochs=10, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for e in range(epochs):
        # --- train ---
        model.train()
        train_correct = train_total = train_loss = 0

        for x, y in train_loader:
            opt.zero_grad()
            out = model(x)
            loss = loss_fn(out, y)
            loss.backward()
            opt.step()

            train_loss += loss.item() * y.size(0)
            train_correct += (out.argmax(1) == y).sum().item()
            train_total += y.size(0)

        # --- val ---
        model.eval()
        val_correct = val_total = val_loss = 0

        with torch.no_grad():
            for x, y in val_loader:
                out = model(x)
                loss = loss_fn(out, y)

                val_loss += loss.item() * y.size(0)
                val_correct += (out.argmax(1) == y).sum().item()
                val_total += y.size(0)

        print(
            f"Epoch {e+1}/{epochs} | "
            f"Train loss {train_loss/train_total:.4f} acc {100*train_correct/train_total:.1f}% | "
            f"Val loss {val_loss/val_total:.4f} acc {100*val_correct/val_total:.1f}%"
        )

In [24]:
batch_size = 32
epochs = 10
hidden_dim = 128
lr = 0.001

models = {}

for emb_type in emb_types:
    print(f"\n{'='*60}\nTraining {emb_type.upper()}\n{'='*60}")
    
    seq_train, label_train = datasets[emb_type]['train']
    seq_val, label_val = datasets[emb_type]['val']
    seq_test, label_test = datasets[emb_type]['test']
    
    train_loader = DataLoader(SentenceDataset(seq_train, label_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(SentenceDataset(seq_val, label_val), batch_size=batch_size)
    test_loader = DataLoader(SentenceDataset(seq_test, label_test), batch_size=batch_size)
    
    embedding_dim = seq_train.shape[2]
    model = RNNClassifier(embedding_dim, hidden_dim)
    train(model, train_loader, val_loader, epochs, lr)
    
    models[emb_type] = {'model': model, 'test_loader': test_loader}



Training W2V
Epoch 1/10 | Train loss 0.6935 acc 49.6% | Val loss 0.6937 acc 50.0%
Epoch 2/10 | Train loss 0.6936 acc 50.3% | Val loss 0.6936 acc 50.0%
Epoch 3/10 | Train loss 0.6935 acc 48.9% | Val loss 0.6934 acc 50.0%
Epoch 4/10 | Train loss 0.6934 acc 49.2% | Val loss 0.6932 acc 50.0%
Epoch 5/10 | Train loss 0.6934 acc 50.1% | Val loss 0.6934 acc 50.0%
Epoch 6/10 | Train loss 0.6933 acc 49.0% | Val loss 0.6932 acc 50.0%
Epoch 7/10 | Train loss 0.6933 acc 49.2% | Val loss 0.6932 acc 50.0%
Epoch 8/10 | Train loss 0.6933 acc 48.9% | Val loss 0.6931 acc 50.0%
Epoch 9/10 | Train loss 0.6933 acc 49.6% | Val loss 0.6932 acc 50.0%
Epoch 10/10 | Train loss 0.6933 acc 49.7% | Val loss 0.6932 acc 50.0%

Training FT
Epoch 1/10 | Train loss 0.6936 acc 49.6% | Val loss 0.6934 acc 50.0%
Epoch 2/10 | Train loss 0.6936 acc 48.4% | Val loss 0.6932 acc 50.0%
Epoch 3/10 | Train loss 0.6934 acc 48.9% | Val loss 0.6932 acc 50.0%
Epoch 4/10 | Train loss 0.6934 acc 48.6% | Val loss 0.6931 acc 50.0%
Epoch 

In [25]:
def test(model: nn.Module, test_loader: DataLoader) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    all_true_labels = []
    all_pred_labels = []
    
    with torch.no_grad():
        for sequences_batch, labels_batch in test_loader:
            logits = model(sequences_batch)
            predictions = logits.argmax(dim=1)
            
            all_true_labels.append(labels_batch.cpu().numpy())
            all_pred_labels.append(predictions.cpu().numpy())
    
    return np.concatenate(all_true_labels), np.concatenate(all_pred_labels)

In [26]:
def evaluate(true_labels: np.ndarray, pred_labels: np.ndarray) -> Dict[str, str]:
    
    print(classification_report(true_labels, pred_labels, target_names=['English', 'French']))
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average='weighted', zero_division=0)
    
    return {
        'Precision': f"{precision:.4f}",
        'Recall': f"{recall:.4f}",
        'F1-Score': f"{f1:.4f}"
    }


In [27]:
results = []

for emb_type in emb_types:
    print(f"\n{'='*60}\nTesting {emb_type.upper()}\n{'='*60}")
    
    true_labels, pred_labels = test(models[emb_type]['model'], models[emb_type]['test_loader'])
    metrics = evaluate(true_labels, pred_labels)
    results.append({'Embedding': emb_type.upper(), **metrics})

results_df = pd.DataFrame(results)
print(f"\n{'='*60}\nSUMMARY\n{'='*60}")
print(results_df.to_string(index=False))



Testing W2V


c:\Users\etulyon1\anaconda3\envs\nlp_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\etulyon1\anaconda3\envs\nlp_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\etulyon1\anaconda3\envs\nlp_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is

              precision    recall  f1-score   support

     English       0.50      1.00      0.67       998
      French       0.00      0.00      0.00       997

    accuracy                           0.50      1995
   macro avg       0.25      0.50      0.33      1995
weighted avg       0.25      0.50      0.33      1995


Testing FT


c:\Users\etulyon1\anaconda3\envs\nlp_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\etulyon1\anaconda3\envs\nlp_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\etulyon1\anaconda3\envs\nlp_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is

              precision    recall  f1-score   support

     English       0.50      1.00      0.67       998
      French       0.00      0.00      0.00       997

    accuracy                           0.50      1995
   macro avg       0.25      0.50      0.33      1995
weighted avg       0.25      0.50      0.33      1995


Testing GLV
              precision    recall  f1-score   support

     English       0.00      0.00      0.00      1000
      French       0.50      1.00      0.67      1000

    accuracy                           0.50      2000
   macro avg       0.25      0.50      0.33      2000
weighted avg       0.25      0.50      0.33      2000


SUMMARY
Embedding Precision Recall F1-Score
      W2V    0.2503 0.5003   0.3336
       FT    0.2503 0.5003   0.3336
      GLV    0.2500 0.5000   0.3333


c:\Users\etulyon1\anaconda3\envs\nlp_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\etulyon1\anaconda3\envs\nlp_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\etulyon1\anaconda3\envs\nlp_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is